# AgroSmart — Fase 1: treino do classificador de sanidade foliar

Treina uma **MobileNetV2** por *transfer learning* sobre um subconjunto de 12 classes do dataset **PlantVillage**, e exporta os arquivos que o app `streamlit` consome.

## Antes de rodar

1. `Ambiente de execução` → `Alterar o tipo de ambiente de execução` → **GPU T4**.
2. `Ambiente de execução` → `Executar tudo`. O treino leva cerca de 20–30 minutos na T4.
3. Ao final, a última célula baixa `agrosmart_modelo.zip`. Descompacte o conteúdo dentro da pasta `modelo/` do projeto.

## Saídas geradas

| Arquivo | Destino |
|---|---|
| `agrosmart_mobilenetv2.keras` | `modelo/` |
| `classes.json` | `modelo/` |
| `matriz_confusao.png`, `curvas_treino.png`, `amostras_treino.png` | `docs/` (entram no relatório) |
| `metricas.json` | `docs/` |

In [ ]:
import json, collections
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

SEMENTE = 42
keras.utils.set_random_seed(SEMENTE)

TAMANHO_ENTRADA = (224, 224)
TAMANHO_LOTE = 32

SAIDA = Path('saida'); SAIDA.mkdir(exist_ok=True)

print('TensorFlow', tf.__version__)
print('GPU disponível:', tf.config.list_physical_devices('GPU') or 'NENHUMA — o treino vai ficar muito lento')

## 1. Catálogo de classes

Cópia fiel de `src/rotulos.py`, para o notebook ser autossuficiente no Colab. **A ordem define os índices 0..11 da camada de saída** — o app valida essa ordem ao carregar o modelo e recusa arquivos incompatíveis.

O casamento com os rótulos do TFDS é feito por fragmentos sobre o nome normalizado (minúsculas, só alfanuméricos), então diferenças de pontuação entre versões do PlantVillage não quebram o filtro.

In [ ]:
VERSAO_CATALOGO = '1.0.0'

CATALOGO = [
    ('apple_healthy',        ['apple', 'healthy'],           'Maçã — Saudável'),
    ('apple_scab',           ['apple', 'scab'],              'Maçã — Sarna-da-macieira'),
    ('corn_healthy',         ['corn', 'healthy'],            'Milho — Saudável'),
    ('corn_common_rust',     ['corn', 'common', 'rust'],     'Milho — Ferrugem comum'),
    ('grape_healthy',        ['grape', 'healthy'],           'Uva — Saudável'),
    ('grape_black_rot',      ['grape', 'black', 'rot'],      'Uva — Podridão-negra'),
    ('potato_healthy',       ['potato', 'healthy'],          'Batata — Saudável'),
    ('potato_early_blight',  ['potato', 'early', 'blight'],  'Batata — Pinta-preta'),
    ('tomato_healthy',       ['tomato', 'healthy'],          'Tomate — Saudável'),
    ('tomato_late_blight',   ['tomato', 'late', 'blight'],   'Tomate — Requeima'),
    ('tomato_septoria',      ['tomato', 'septoria'],         'Tomate — Mancha de septória'),
    ('tomato_spider_mites',  ['tomato', 'spider', 'mite'],   'Tomate — Ácaro-rajado'),
]

IDS = [linha[0] for linha in CATALOGO]
NOMES_PT = [linha[2] for linha in CATALOGO]
N_CLASSES = len(IDS)

def normalizar(nome):
    return ''.join(c for c in nome.lower() if c.isalnum())

def casar_rotulo(nome_dataset):
    alvo = normalizar(nome_dataset)
    casados = [id_ for id_, padroes, _ in CATALOGO
               if all(normalizar(p) in alvo for p in padroes)]
    if len(casados) > 1:
        raise ValueError(f'Rótulo ambíguo: {nome_dataset!r} casa com {casados}')
    return casados[0] if casados else None

print(f'{N_CLASSES} classes no catálogo v{VERSAO_CATALOGO}')

## 2. Carga do PlantVillage e filtro das 12 classes

O download são ~830 MB (54.303 imagens, 38 classes) e fica em cache no disco da sessão.

In [ ]:
ds_bruto, info = tfds.load('plant_village', split='train', as_supervised=True,
                           with_info=True, shuffle_files=False)

nomes_dataset = info.features['label'].names
print(f'{len(nomes_dataset)} classes no dataset original, {info.splits["train"].num_examples} imagens\n')

# indice original do TFDS -> indice 0..11 do AgroSmart (ou -1 se descartada)
mapa = {}
for indice, nome in enumerate(nomes_dataset):
    id_classe = casar_rotulo(nome)
    if id_classe is not None:
        mapa[indice] = IDS.index(id_classe)
        print(f'  {nome:<55} -> [{mapa[indice]:2d}] {NOMES_PT[mapa[indice]]}')

assert len(mapa) == N_CLASSES, (
    f'Esperava casar {N_CLASSES} classes, casei {len(mapa)}. '
    'Os nomes dos rótulos do TFDS mudaram — revise os padrões do CATALOGO.'
)
assert sorted(mapa.values()) == list(range(N_CLASSES)), 'Duas classes do dataset caíram no mesmo índice.'
print(f'\nOK: {len(mapa)}/{len(nomes_dataset)} classes selecionadas.')

## 3. Divisão treino / validação / teste

O PlantVillage é armazenado **agrupado por classe**, em blocos contíguos. Isso permite uma divisão estratificada sem embaralhar nada e sem consumir memória: basta usar a posição do exemplo módulo 20.

| Resto de `i % 20` | Destino | Proporção |
|---|---|---|
| 0–2 | teste | 15% |
| 3–5 | validação | 15% |
| 6–19 | treino | 70% |

Como o corte percorre cada bloco de classe uniformemente, cada classe é fatiada na mesma proporção. A célula seguinte comprova isso numericamente — e a tabela vale como evidência metodológica no relatório.

In [ ]:
# Lê apenas os rótulos (SkipDecoding evita decodificar 54 mil JPEGs à toa)
ds_rotulos = tfds.load('plant_village', split='train', shuffle_files=False,
                       decoders={'image': tfds.decode.SkipDecoding()})
sequencia = np.fromiter(
    (int(exemplo['label']) for exemplo in ds_rotulos.as_numpy_iterator()),
    dtype=np.int64,
)

manter = np.isin(sequencia, list(mapa.keys()))
rotulos_filtrados = np.array([mapa[valor] for valor in sequencia[manter]])
posicoes = np.arange(len(rotulos_filtrados))

destino = np.where(posicoes % 20 < 3, 'teste',
           np.where(posicoes % 20 < 6, 'validação', 'treino'))

distribuicao = pd.crosstab(
    pd.Series([NOMES_PT[r] for r in rotulos_filtrados], name='classe'),
    pd.Series(destino, name='conjunto'),
)[['treino', 'validação', 'teste']]
distribuicao['total'] = distribuicao.sum(axis=1)
distribuicao['% treino'] = (distribuicao['treino'] / distribuicao['total'] * 100).round(1)

N_TREINO = int((destino == 'treino').sum())
print(f'Total filtrado: {len(rotulos_filtrados)} imagens\n')
display(distribuicao)

In [ ]:
tabela_mapa = tf.constant([mapa.get(i, -1) for i in range(len(nomes_dataset))], dtype=tf.int64)

def remapear(imagem, rotulo):
    return imagem, tf.gather(tabela_mapa, rotulo)

ds_filtrado = (ds_bruto
               .map(remapear, num_parallel_calls=tf.data.AUTOTUNE)
               .filter(lambda _imagem, rotulo: rotulo >= 0)
               .enumerate())

def fatiar(inicio, fim):
    return (ds_filtrado
            .filter(lambda i, _par: tf.logical_and(i % 20 >= inicio, i % 20 < fim))
            .map(lambda _i, par: par, num_parallel_calls=tf.data.AUTOTUNE))

def preparar(imagem, rotulo):
    # Mantém a escala bruta 0..255: a normalização do MobileNetV2 acontece
    # dentro do modelo, numa camada Rescaling, para treino e inferência não
    # divergirem.
    imagem = tf.image.resize(imagem, TAMANHO_ENTRADA)
    return tf.cast(imagem, tf.float32), rotulo

def montar(dataset, embaralhar=False):
    dataset = dataset.map(preparar, num_parallel_calls=tf.data.AUTOTUNE)
    if embaralhar:
        dataset = dataset.shuffle(2048, seed=SEMENTE, reshuffle_each_iteration=True)
    return dataset.batch(TAMANHO_LOTE).prefetch(tf.data.AUTOTUNE)

ds_teste     = montar(fatiar(0, 3))
ds_validacao = montar(fatiar(3, 6))
ds_treino    = montar(fatiar(6, 20), embaralhar=True)

print('Pipelines prontos.')

## 4. Aumento de dados

Aplicado **apenas ao treino**. É a peça mais importante deste notebook: as imagens do PlantVillage são folhas destacadas sobre fundo cinza uniforme de laboratório, e sem variação forte o modelo aprende o fundo em vez da lesão. Rotação, zoom, brilho e contraste aproximam as amostras de uma foto de celular no campo.

In [ ]:
aumento = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical', seed=SEMENTE),
    layers.RandomRotation(0.20, fill_mode='reflect', seed=SEMENTE),
    layers.RandomZoom(0.15, fill_mode='reflect', seed=SEMENTE),
    layers.RandomContrast(0.20, seed=SEMENTE),
    layers.RandomBrightness(0.20, value_range=(0, 255), seed=SEMENTE),
], name='aumento')

ds_treino_aumentado = ds_treino.map(
    lambda imagem, rotulo: (aumento(imagem, training=True), rotulo),
    num_parallel_calls=tf.data.AUTOTUNE,
).prefetch(tf.data.AUTOTUNE)

# Amostra visual — vai para o relatório
imagens, rotulos_lote = next(iter(ds_treino_aumentado))
figura, eixos = plt.subplots(3, 6, figsize=(15, 8))
for eixo, imagem, rotulo in zip(eixos.ravel(), imagens, rotulos_lote):
    eixo.imshow(np.clip(imagem.numpy(), 0, 255).astype('uint8'))
    eixo.set_title(NOMES_PT[int(rotulo)], fontsize=7)
    eixo.axis('off')
figura.suptitle('Amostras do conjunto de treino após aumento de dados', fontsize=12)
figura.tight_layout()
figura.savefig(SAIDA / 'amostras_treino.png', dpi=140, bbox_inches='tight')
plt.show()

## 5. Modelo

Cópia fiel de `src/arquitetura.py`. A camada `Rescaling(1/127.5, offset=-1)` reproduz o `preprocess_input` do MobileNetV2 dentro do próprio modelo, o que elimina a possibilidade de o pré-processamento do app divergir do pré-processamento do treino.

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=(*TAMANHO_ENTRADA, 3), include_top=False, weights='imagenet')
base.trainable = False

entradas = keras.Input(shape=(*TAMANHO_ENTRADA, 3), name='imagem')
x = layers.Rescaling(1 / 127.5, offset=-1, name='normalizacao')(entradas)
x = base(x, training=False)   # BatchNorm sempre em modo inferência, inclusive no fine-tune
x = layers.GlobalAveragePooling2D(name='pooling')(x)
x = layers.Dropout(0.3, name='dropout')(x)
saidas = layers.Dense(N_CLASSES, activation='softmax', name='diagnostico')(x)

modelo = keras.Model(entradas, saidas, name='agrosmart_mobilenetv2')
modelo.summary()

## 6. Fase A — treino do classificador com a base congelada

In [ ]:
modelo.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

parada = keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1)

historico_a = modelo.fit(
    ds_treino_aumentado, validation_data=ds_validacao,
    epochs=8, callbacks=[parada],
)

## 7. Fase B — ajuste fino das últimas camadas

Descongela as 30 últimas camadas da base com taxa de aprendizado 100× menor. Sem essa redução, os gradientes do classificador recém-treinado destruiriam as representações do ImageNet.

In [ ]:
base.trainable = True
for camada in base.layers[:-30]:
    camada.trainable = False

modelo.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
treinaveis = sum(1 for c in base.layers if c.trainable)
print(f'{treinaveis} de {len(base.layers)} camadas da base liberadas para ajuste.\n')

historico_b = modelo.fit(
    ds_treino_aumentado, validation_data=ds_validacao,
    epochs=6, callbacks=[parada],
)

In [ ]:
def juntar(chave):
    return historico_a.history[chave] + historico_b.history[chave]

corte = len(historico_a.history['loss'])
figura, (esq, dir_) = plt.subplots(1, 2, figsize=(13, 4.5))

for eixo, chave, titulo in ((esq, 'accuracy', 'Acurácia'), (dir_, 'loss', 'Perda')):
    eixo.plot(juntar(chave), label='treino')
    eixo.plot(juntar(f'val_{chave}'), label='validação')
    eixo.axvline(corte - 0.5, color='gray', linestyle='--', linewidth=1)
    eixo.annotate('início do ajuste fino', xy=(corte - 0.5, eixo.get_ylim()[0]),
                  fontsize=8, color='gray', rotation=90, va='bottom', ha='right')
    eixo.set_title(titulo); eixo.set_xlabel('época'); eixo.legend(); eixo.grid(alpha=0.3)

figura.suptitle('Evolução do treino — Fase A (base congelada) e Fase B (ajuste fino)')
figura.tight_layout()
figura.savefig(SAIDA / 'curvas_treino.png', dpi=140, bbox_inches='tight')
plt.show()

## 8. Avaliação no conjunto de teste

O conjunto de teste nunca foi visto pelo modelo nem influenciou a parada antecipada.

In [ ]:
y_verdadeiro = np.concatenate([rotulo.numpy() for _imagem, rotulo in ds_teste])
y_probabilidade = modelo.predict(ds_teste, verbose=1)
y_previsto = y_probabilidade.argmax(axis=1)

acuracia = float((y_previsto == y_verdadeiro).mean())
print(f'\nAcurácia no teste: {acuracia:.4f}  ({len(y_verdadeiro)} imagens)\n')
print(classification_report(y_verdadeiro, y_previsto, target_names=NOMES_PT, digits=3))

In [ ]:
matriz = confusion_matrix(y_verdadeiro, y_previsto, normalize='true')

figura, eixo = plt.subplots(figsize=(10, 8.5))
imagem = eixo.imshow(matriz, cmap='YlGn', vmin=0, vmax=1)
eixo.set_xticks(range(N_CLASSES), NOMES_PT, rotation=45, ha='right', fontsize=8)
eixo.set_yticks(range(N_CLASSES), NOMES_PT, fontsize=8)
eixo.set_xlabel('Previsto'); eixo.set_ylabel('Real')
eixo.set_title(f'Matriz de confusão normalizada — acurácia {acuracia:.1%}')

for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if matriz[i, j] >= 0.01:
            eixo.text(j, i, f'{matriz[i, j]:.2f}', ha='center', va='center', fontsize=7,
                      color='white' if matriz[i, j] > 0.6 else 'black')

figura.colorbar(imagem, fraction=0.046)
figura.tight_layout()
figura.savefig(SAIDA / 'matriz_confusao.png', dpi=140, bbox_inches='tight')
plt.show()

### Cobertura do limiar de rejeição

Quantas predições **corretas** o limiar de 0,60 descartaria por falta de confiança? Se esse número for alto, o limiar do app está apertado demais e precisa ser recalibrado.

In [ ]:
LIMIAR = 0.60
confianca = y_probabilidade.max(axis=1)
acertou = y_previsto == y_verdadeiro

print(f'Confiança média — acertos: {confianca[acertou].mean():.3f} | erros: {confianca[~acertou].mean():.3f}')
print(f'Acertos descartados pelo limiar de {LIMIAR:.0%}: '
      f'{int((acertou & (confianca < LIMIAR)).sum())} de {int(acertou.sum())} '
      f'({(acertou & (confianca < LIMIAR)).mean():.2%} do total)')
print(f'Erros barrados pelo limiar: '
      f'{int((~acertou & (confianca < LIMIAR)).sum())} de {int((~acertou).sum())}')

## 9. Exportação

In [ ]:
caminho_modelo = SAIDA / 'agrosmart_mobilenetv2.keras'
modelo.save(caminho_modelo)

metadados = {
    'versao_catalogo': VERSAO_CATALOGO,
    'arquitetura': 'MobileNetV2',
    'tamanho_entrada': list(TAMANHO_ENTRADA),
    'treinado_em': datetime.now().isoformat(timespec='seconds'),
    'acuracia_teste': round(acuracia, 4),
    'ids': IDS,
}
(SAIDA / 'classes.json').write_text(json.dumps(metadados, ensure_ascii=False, indent=2), encoding='utf-8')

relatorio = classification_report(y_verdadeiro, y_previsto, target_names=NOMES_PT,
                                  digits=4, output_dict=True)
(SAIDA / 'metricas.json').write_text(json.dumps({
    'acuracia_teste': acuracia,
    'imagens_teste': int(len(y_verdadeiro)),
    'imagens_treino': int(N_TREINO),
    'limiar_confianca': LIMIAR,
    'por_classe': relatorio,
}, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Modelo: {caminho_modelo.stat().st_size / 1e6:.1f} MB')
for arquivo in sorted(SAIDA.iterdir()):
    print(' -', arquivo.name)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('agrosmart_modelo', 'zip', SAIDA)
files.download('agrosmart_modelo.zip')

print('Descompacte no projeto:')
print('  agrosmart_mobilenetv2.keras, classes.json  ->  modelo/')
print('  *.png, metricas.json                       ->  docs/')

## 10. Teste de campo — a parte que vale nota no relatório

A acurácia medida acima vale **só para fotos no padrão do PlantVillage**: folha destacada, centralizada, sobre fundo uniforme de laboratório. Fotos reais de celular trazem sombra, fundo com solo e outras folhas, foco irregular e ângulo — condições que o modelo nunca viu.

Rode a célula abaixo com 10 fotos tiradas pela equipe (tomate, batata, uva, milho ou maçã, com o diagnóstico real conhecido) e **compare as duas acurácias no relatório**. Uma queda expressiva é o resultado esperado e documentá-la demonstra rigor — muito mais do que apresentar apenas o número de laboratório.

Aproveite para incluir uma foto de roseira: ela deve cair abaixo do limiar e virar *indeterminado*.

In [ ]:
from google.colab import files as upload_files
from PIL import Image, ImageOps
import io

enviados = upload_files.upload()

linhas = []
for nome, conteudo in enviados.items():
    imagem = ImageOps.exif_transpose(Image.open(io.BytesIO(conteudo))).convert('RGB')
    tensor = np.asarray(imagem.resize(TAMANHO_ENTRADA), dtype='float32')[None, ...]
    probabilidade = modelo.predict(tensor, verbose=0)[0]
    melhor = int(probabilidade.argmax())
    linhas.append({
        'arquivo': nome,
        'diagnóstico': NOMES_PT[melhor] if probabilidade[melhor] >= LIMIAR else 'INDETERMINADO',
        'confiança': round(float(probabilidade[melhor]), 3),
        'segunda opção': NOMES_PT[int(probabilidade.argsort()[-2])],
    })

display(pd.DataFrame(linhas))